# TEST INFERENCE PIPELINE

In [ ]:
from pathlib import Path
import numpy as np
import torch
import html
import re

from torch.utils.data import DataLoader
from emma import EmmaDataset, EmmaSearcher, SearchQuery
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate

from gliner import GLiNER

import utils.logging_ as logging

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass"], limit=64)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
def clean_title(title: str) -> str:
    title = html.unescape(title)
    title = re.sub(r"\s+", " ", title)
    title = title.replace("｜", "|")
    title = title.strip()
    return title

def my_transform(asin: str, rec: dict):
    title = clean_title(ds.get_title(rec))
    category = ds.get_category_path(rec)

    return {
        "title": title,
        "category": category,
        "asin": asin,
    }

In [ ]:
loader = DataLoader(
    EmmaMapDataset(ds, asins, transform=my_transform),
    batch_size=1, shuffle=False, collate_fn=emma_collate,
)

## > MODEL

In [ ]:
gliner_root = None  # e.g. Path("/path/to/GLiNER")

if gliner_root is None:
    raise ValueError("Please set `gliner_root` to the path of the GLiNER model.")

gliner_root = Path(gliner_root)

if not gliner_root.exists():
    raise FileNotFoundError(f"Model path does not exist: {gliner_root}")

In [ ]:
model = GLiNER.from_pretrained(
    gliner_root,
    map_location="cuda",
    load_tokenizer=True,
)

model.to(torch.float16)  # fp16 half-precision
model.compile()          # torch.compile with dynamic shapes

## > INFERENCE

In [ ]:
# Warmup
_ = model.inference(
    ["Warmup title for compilation"],
    ["product type"],
    batch_size=1,
    threshold=0.3,
)

In [ ]:
# labels = ["main product object", "secondary product object", "product attribute", "compatible product", "brand", "size", "application"]
# labels = ["main item being sold"]
labels = ["generic product type"]

In [ ]:
for batch_idx, batch in enumerate(loader):
    title = batch['title']
    category = batch['category']

    predictions = model.inference(
        title,
        labels,
        batch_size=1,
        threshold=0.3,
    )

    logging.progress_bar(batch_idx, len(loader), 1, 1)
    print("")

    print("\nTITLE:", title)
    print("\nCATEGORY:", category)
    for e in predictions[0]:
        if predictions is None:
            print("No Predictions")
        else:
            print(e["text"], "=>", e["label"], f"({e['score']:.3f})")
    print("")
        
print("Done")